In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import glob
import warnings
warnings.filterwarnings('ignore')

## Test Regression Head (sh)

### Combined Features (Geo-encoded + Visual)

In [2]:
base_folder = 'results/*'
files = sorted(glob.glob(base_folder))[1:]
mae_table = {}
files

['results/regression_comparison',
 'results/split_spatialL_1_loc_locenc',
 'results/split_spatialL_2_loc_locenc',
 'results/split_spatialL_3_loc_locenc',
 'results/split_spatialL_4_loc_locenc',
 'results/split_spatialL_5_loc_locenc']

In [4]:
xgb_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', xgb.XGBRegressor(
            n_estimators=1000, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
        ))
])

In [29]:
mae_table['xgboost'] = {}
model = xgb_pipeline

for fold in range(1, 6):
    fold_folder = files[fold-1]
    print(f'\n Processing Fold {fold}: {fold_folder}')
    
    X_train = pd.read_csv(f"{fold_folder}/X_train_combined.csv").values
    X_test = pd.read_csv(f"{fold_folder}/X_test_combined.csv").values
    y_train = pd.read_csv(f"{fold_folder}/train_predictions_with_metadata.csv")['target'].values
    y_test = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")['target'].values
    
    print(f"  Loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")

    model.fit(X_train, y_train)
    test_predictions = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_predictions)
    
    mae_table['xgboost'][f'fold {fold}'] = test_mae
    

    print(f'xgboost: test_mae: {test_mae:.4f}')

print(np.mean(list(mae_table["xgboost"].values())))
    
    


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
xgboost: test_mae: 0.1818

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
xgboost: test_mae: 0.1772

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
xgboost: test_mae: 0.1844

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
xgboost: test_mae: 0.1802

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
xgboost: test_mae: 0.1850
0.18172151730839073


In [5]:
lightgbm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lgb', lgb.LGBMRegressor(
        n_estimators=1000, max_depth=8, learning_rate=0.1,
        feature_fraction=0.8, bagging_fraction=0.8, random_state=42, verbosity=-1
    ))
])

In [20]:
mae_table['light_gbm'] = {}
model = lightgbm_pipeline
for fold in range(1, 6):
    fold_folder = files[fold-1]
    print(f'\n Processing Fold {fold}: {fold_folder}')
    
    X_train = pd.read_csv(f"{fold_folder}/X_train_combined.csv").values
    X_test = pd.read_csv(f"{fold_folder}/X_test_combined.csv").values
    y_train = pd.read_csv(f"{fold_folder}/train_predictions_with_metadata.csv")['target'].values
    y_test = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")['target'].values
    
    print(f"  Loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")

    model.fit(X_train, y_train)
    test_predictions = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_predictions)

    mae_table['light_gbm'][f'fold {fold}'] = test_mae
    
    print(f'light_gbm: test_mae: {test_mae:.4f}')

print(f'Average: {np.mean(list(mae_table["light_gbm"].values()))}')
    
    
    


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
light_gbm: test_mae: 0.1819

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
light_gbm: test_mae: 0.1767

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
light_gbm: test_mae: 0.1820

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
light_gbm: test_mae: 0.1809

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
light_gbm: test_mae: 0.1837


AttributeError: 'dict_values' object has no attribute 'mean'

In [25]:
np.mean(list(mae_table["light_gbm"].values()))

0.1810708776254164

In [27]:
random_forest_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(
        n_estimators=500, max_depth=15, min_samples_split=5,
        min_samples_leaf=2, random_state=42, n_jobs=-1
    ))
])

In [ ]:
mae_table['random_forest'] = {}
model = random_forest_pipeline
for fold in range(1, 6):
    fold_folder = files[fold-1]
    print(f'\n Processing Fold {fold}: {fold_folder}')
    
    X_train = pd.read_csv(f"{fold_folder}/X_train_combined.csv").values
    X_test = pd.read_csv(f"{fold_folder}/X_test_combined.csv").values
    y_train = pd.read_csv(f"{fold_folder}/train_predictions_with_metadata.csv")['target'].values
    y_test = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")['target'].values
    
    print(f"  Loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")

    model.fit(X_train, y_train)
    test_predictions = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_predictions)

    mae_table['random_forest'][f'fold {fold}'] = test_mae
    
    print(f'random forest: test_mae: {test_mae:.4f}')

print(f'Average: {np.mean(list(mae_table["random_forest"].values()))}')
    
    

### Visual-only Features

In [12]:
base_folder = 'results/*'
files = sorted(glob.glob(base_folder))[1:]
mae_table_visual = {}
mae_table_visual['light_gbm'] = {}
files

['results/split_spatialL_1_loc_locenc',
 'results/split_spatialL_2_loc_locenc',
 'results/split_spatialL_3_loc_locenc',
 'results/split_spatialL_4_loc_locenc',
 'results/split_spatialL_5_loc_locenc']

In [13]:
model = lightgbm_pipeline
for fold in range(1, 6):
    fold_folder = files[fold-1]
    print(f'\n Processing Fold {fold}: {fold_folder}')
    
    X_train = pd.read_csv(f"{fold_folder}/X_train_visual.csv").values
    X_test = pd.read_csv(f"{fold_folder}/X_test_visual.csv").values
    y_train = pd.read_csv(f"{fold_folder}/train_predictions_with_metadata.csv")['target'].values
    y_test = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")['target'].values
    
    print(f"  Loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")

    model.fit(X_train, y_train)
    test_predictions = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_predictions)

    mae_table_visual['light_gbm'][f'fold {fold}'] = test_mae
    
    print(f'light_gbm: test_mae: {test_mae:.4f}')

print(f'Average: {np.mean(list(mae_table_visual["light_gbm"].values()))}')


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
light_gbm: test_mae: 0.2134

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
light_gbm: test_mae: 0.2114

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
light_gbm: test_mae: 0.2155

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
light_gbm: test_mae: 0.2116

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
light_gbm: test_mae: 0.2191
Average: 0.21419448467637475


In [15]:
mae_table_visual['xgboost'] = {}
model = xgb_pipeline

for fold in range(1, 6):
    fold_folder = files[fold-1]
    print(f'\n Processing Fold {fold}: {fold_folder}')
    
    X_train = pd.read_csv(f"{fold_folder}/X_train_visual.csv").values
    X_test = pd.read_csv(f"{fold_folder}/X_test_visual.csv").values
    y_train = pd.read_csv(f"{fold_folder}/train_predictions_with_metadata.csv")['target'].values
    y_test = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")['target'].values
    
    print(f"  Loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")

    model.fit(X_train, y_train)
    test_predictions = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_predictions)
    
    mae_table_visual['xgboost'][f'fold {fold}'] = test_mae

    print(f'xgboost: test_mae: {test_mae:.4f}')

print(np.mean(list(mae_table_visual["xgboost"].values())))


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
xgboost: test_mae: 0.2137

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
xgboost: test_mae: 0.2130

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
xgboost: test_mae: 0.2175

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
xgboost: test_mae: 0.2126

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
xgboost: test_mae: 0.2186
0.21508324470107493


## Test Regression Head (sh+siren)

In [7]:
base_folder = 'results/split_spatialL_*_sh_siren'
files = sorted(glob.glob(base_folder))
mae_table_sh_siren = {}
files

['results/split_spatialL_1_sh_siren',
 'results/split_spatialL_2_sh_siren',
 'results/split_spatialL_3_sh_siren',
 'results/split_spatialL_4_sh_siren',
 'results/split_spatialL_5_sh_siren']

In [ ]:
mae_table_sh_siren['light_gbm'] = {}
model = lightgbm_pipeline
for fold in range(1, 6):
    fold_folder = files[fold-1]
    print(f'\n Processing Fold {fold}: {fold_folder}')
    
    X_train = pd.read_csv(f"{fold_folder}/X_train_combined.csv").values
    X_test = pd.read_csv(f"{fold_folder}/X_test_combined.csv").values
    y_train = pd.read_csv(f"{fold_folder}/train_predictions_with_metadata.csv")['target'].values
    y_test = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")['target'].values
    
    print(f"  Loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")

    model.fit(X_train, y_train)
    test_predictions = model.predict(X_test)
    test_mae = mean_absolute_error(y_test, test_predictions)

    mae_table_sh_siren['light_gbm'][f'fold {fold}'] = test_mae
    
    print(f'light_gbm: test_mae: {test_mae:.4f}')

print(f'Average: {np.mean(list(mae_table_sh_siren["light_gbm"].values()))}')


 Processing Fold 1: results/split_spatialL_1_sh_siren
  Loaded: 10602 train, 2713 test samples
